## Step 1: Setup and paths
1. Set the folder that holds your videos.
2. Choose how many frames you want per video.
3. Run the cell to install and import dependencies.

In [ ]:
from pathlib import Path
import cv2
import numpy as np

In [ ]:
# Config
video_folder = Path("/path/to/videos")  # Folder containing your videos
frames_per_video = 25  # X photos per video
image_ext = ".jpg"
random_seed = 812
train_split = 0.8

# Output dataset folders
dataset_root = Path("/path/to/dataset")
images_train_dir = dataset_root / "images" / "train"
images_val_dir = dataset_root / "images" / "val"
labels_train_dir = dataset_root / "labels" / "train"
labels_val_dir = dataset_root / "labels" / "val"

for d in [images_train_dir, images_val_dir, labels_train_dir, labels_val_dir]:
    d.mkdir(parents=True, exist_ok=True)

## Step 2: Extract frames from videos
1. Put your videos in the folder you set above.
2. Run the next cell to extract frames into per-video subfolders.
3. Annotate those frames with your labeling tool (YOLO format).

In [ ]:
def extract_frames_from_video(video_path: Path, frames_count: int, out_dir: Path, ext: str = ".jpg"):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Skipping {video_path.name}: cannot open")
        return 0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        print(f"Skipping {video_path.name}: no frames")
        cap.release()
        return 0
    out_dir.mkdir(parents=True, exist_ok=True)
    indices = np.linspace(0, total_frames - 1, frames_count, dtype=int)
    saved = 0
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok:
            continue
        out_file = out_dir / f"{video_path.stem}_f{idx:06d}{ext}"
        cv2.imwrite(str(out_file), frame)
        saved += 1
    cap.release()
    return saved

video_files = sorted([p for p in video_folder.iterdir() if p.suffix.lower() in {".mp4", ".mov", ".avi", ".mkv"}])
if not video_files:
    print("No videos found. Check video_folder.")
else:
    for video_path in video_files:
        out_dir = video_path.parent / f"{video_path.stem}_frames"
        saved = extract_frames_from_video(video_path, frames_per_video, out_dir, image_ext)
        print(f"{video_path.name}: saved {saved} frames to {out_dir}")